# Tutorial 08 — Governed execution: story, config, and live checks

This is a **guided lab**, not a dump of printouts. Each block has a short **story** (why the layer
exists), a **knob** you can edit (`USER_*`, overlays, prompts), and **stdout** you read like an
operator would read logs and policy traces.

**What you will understand**

1. **Ingress (pre-model)** — text enters the system; gates can **deny / escalate** before any model
   spend. You configure patterns and profiles like tenant overlay keys.
2. **Tool policy (risk + tenant overlay)** — every tool intent passes **policy** (`before_tool_call`).
   **Allow** lets execution continue; **deny / escalate** return structured **blocked** envelopes instead
   of running your Python handler.
3. **Execution mode** — for **low-risk, non-state-changing** work, the stack may route **provider-native**
   (model/SDK path). For **high-risk or state-changing** work, eXo-brain **forces deterministic** execution:
   your registered **handler** runs inside `DeterministicToolExecutor`, so the model only sees **typed
   `ToolResult`**, not raw side effects. That is how governance stays **accurate and auditable**.

**How to run it**

- Run **top to bottom** the first time so `policy_overlay`, `registry`, `executor`, and `chain` exist.
- **Parts 1–7** need **no API key** (local policy, ingress, stub orchestrator, and `planned_tool_call`).
- **Part 8** is optional: set **`OPENAI_API_KEY`** for live contrasts (**8.1 setup**, then **§1–§4** cells).
  Governed proofs use **`planned_tool_call`** (same as Part 7); **8.6** prints the summary table.

**Requires:** `exo-brain-core-contracts` — in-tree under `packages/eXo_adapters/...` (first code cell adds
`src` to `sys.path` when present) or `pip install -r requirements.txt`.

**Further reading:** `docs/architecture/governed-execution-pipeline.md` (ordering of ingress, orchestrator,
policy, deterministic tools on the full API path).

## For non-technical readers

You do **not** need to read Python to get value from this lab. Use this box as your **executive path**,
then skim each Part’s **Story** heading (skip code if you prefer).

### The problem in one sentence

Teams want helpful AI — but **not** at the price of leaking secrets, triggering dangerous actions, or
racking up model and tool spend with **no trace** of who allowed what.

### What you gain (business language)

| You gain | What it feels like day to day |
|----------|------------------------------|
| **Safety** | Risky or sensitive input can be **stopped or sent for review** *before* the model runs. |
| **Control** | Rules decide **what may run** — not vibes from the model. |
| **Predictability** | Important outcomes can follow **repeatable** logic you can test, not one-off guesses. |
| **Proof** | Allow/deny decisions carry **reasons** you can show support, security, or auditors. |
| **Cost discipline** | Problems caught **early** mean fewer wasted tokens and tool calls. |

### The “three numbers” proof (Part 4 + optional Part 8)

Imagine asking for **11 + 33**. A model can say **44** from memory. In this lab, the real answer is
**11 + 33 + a secret third addend** only your server knows — plus a **proof code** the model cannot invent.
Part **4** prints **`[PASS] Part 4 local proof`** when handler JSON matches your kernel. Part **8** §3
prints **`§3 VERIFICATION (governed): PASS`** when **`planned_tool_call`** shows **`tool_progress` completed**
*and* the assistant cites kernel **sum** + **proof_token** (not mental math).

### What you will *see* when someone runs the cells

- **Healthy path:** words like *allow*, *completed*, or a clear numeric result from a safe tool.
- **Governance doing its job:** *deny*, *blocked*, *escalate*, or a short **reason code** — that is the
  product **protecting you**, not a random error.

### Two ways to use this notebook

1. **Executive path (~3 minutes):** this box → **Map** table below → each Part’s **Story** only.
2. **Hands-on path (~20 minutes):** run **top to bottom**; tweak **Your task** knobs and watch stdout.
   **Part 8** is optional and is the only part that may charge a small OpenAI fee if a key is set.

### Jargon cheat sheet (plain words ↔ what engineers say)

| Engineers say | You can picture |
|-----------------|------------------|
| Ingress | The **door** that reads the message **before** the AI. |
| Policy / risk gates | **Automatic rules** for safe vs risky actions. |
| Deterministic tools | The work ran in **our** code path so the **answer is checkable**. |
| Tenant overlay | **Extra rules for one customer** without changing everyone else’s defaults. |

With an API key, **Part 8** runs short **governed vs raw** comparisons (ingress, blocked tool, proof math,
optional calc). **`§N VERIFICATION (governed): PASS/FAIL`** lines report each section; **§2–§4** use
**`planned_tool_call`** (same as Part 7) so governed proofs do **not** depend on the model choosing tools.
Use **`NB_LIVE_*`** env flags to skip sections and save tokens; CI runs this notebook **without** a key
(Part 8 prints skip).

## Beginner checklist — read this once

1. **Run cells from the top** the first time (bootstrap → Part 1 → …). Later you can jump back to any
   **Part** after the variables it needs exist (`policy_risk`, `policy_overlay`, `registry`, `executor`,
   `chain`).
2. Each **Part** has three cues: **Story** (why), **Your task** (what to edit), **Reading stdout** (what
   good looks like). If stdout confuses you, re-read **Story** for that part only.
3. **“With vs without”** appears in several code cells: the notebook prints **two** behaviours side by
   side (strict vs relaxed policy, overlay on vs off, direct Python vs governed executor). That is
   intentional so you see *what the framework adds*.
4. **Part 8** is the only part that may charge your OpenAI account. If you skip the key, you still learn
   the full local story in Parts 1–7. With a key, set **`NB_LIVE_MATH=0`** (etc.) to turn off individual
   live blocks — see the Part 8 **Cost controls** table.

**Typical first run:** ~10–20 minutes without an API key (includes **Checkpoint** + divide-by-zero demo);
add a few more minutes with Part 8 enabled (fewer if you disable some **`NB_LIVE_*`** flags).

## Map — where you are in the stack

| Stage | You configure (examples) | This notebook |
|-------|--------------------------|----------------|
| Ingress | `INGRESS_OVERLAY`, profiles, custom rules | **Part 6** (and **Part 8** before a live call) |
| Tool policy | `USER_RISK`, `USER_OVERLAY`, tenant id on `ToolCallContext` | **Parts 1–3** |
| Deterministic tools | `USER_TOOLS`; **`safe_add_proven`** (3-operand sum + proof) and **`calculate_result`** | **Part 4** |
| Execution mode | Capability map + policy `enforced_mode` | **Part 5** |
| Orchestrator stream | `planned_tool_call` (stub) or live adapter | **Parts 7–8** |

**Integrator note:** Calling `Orchestrator.run_turn` directly **skips** HTTP-only steps (some entitlements,
budgets). Here we **explicitly** run ingress before Part 8 to mirror the *spirit* of the pipeline doc;
production traffic should still go through **`src/api/routers/turns.py`** (SSE/WebSocket turn execution)
when you integrate — that router applies the full ingress + orchestration stack for real tenants.

**CI:** On pull requests touching `notebooks/**`, CI executes this notebook with **`nbconvert`** (no API
key; Part 8 prints skip). If execution fails, fix **`notebooks/build_tutorials.py`** and regenerate
**`tutorial_08_*.ipynb`**.

## Orientation — table of contents and pipeline (read once)

**Rough time per part** (reading Story + running code; Part 8 adds a few minutes if a key is set):

| Part | Topic | ~Time |
|------|--------|------|
| Bootstrap | paths, `.env` | 1 min |
| 1–2 | Risk gates + synthetic probes | 2–4 min |
| 3 | Tenant overlay (**DENY** vs **ESCALATE** cue) | 2 min |
| 4 | Registry tools + `run_tool` + divide-by-zero error | 4–6 min |
| 5 | Execution mode sweep | 2 min |
| 6 | Ingress gate chain | 3 min |
| **Checkpoint** | verify globals before orchestrator | 30 s |
| 7 | Stub orchestrator stream | 2 min |
| 8 | Live contrasts (optional `$`) | 3–8 min |

**Happy-path pipeline** (this notebook mirrors the middle layers; HTTP adds more gates upstream):

```mermaid
flowchart LR
  U[User text] --> I[Ingress gate chain]
  I -->|ALLOW| O[Orchestrator.run_turn]
  I -->|DENY / ESCALATE| X[Stop before model]
  O --> R[Runtime adapter]
  R --> P[Policy before_tool_call]
  P -->|DENY / ESCALATE| B[Blocked envelope]
  P -->|ALLOW| E[DeterministicToolExecutor]
  E --> H[Your Python handler]
  H --> T[ToolResult to model]
```

In **Cursor / VS Code** and on **GitHub**, the diagram renders from the `mermaid` fence. Plain Jupyter may
show the fence as text unless a Mermaid extension is installed — the **ASCII** takeaway is still:
**ingress → orchestrator → policy → executor → handler**.

## Story — Why “deterministic tools” help the agent answer correctly

The model proposes **names and arguments** for tools. **Governance** decides whether that proposal
may run, and **how** it runs:

- **Deterministic path:** your Python **handler** runs in the executor. The model receives a
  **`ToolResult`** with stable fields (`status`, `error`, audit correlation). Side effects match what
  you coded — not what the SDK guessed.
- **Provider-native path:** the adapter may let the Agents SDK continue the tool loop. That is useful
  for low-risk flows when policy and capability maps agree — but it is **not** where you want silent
  writes or high-risk actions.

So: **deterministic tools do not “make the LLM smarter”** — they **bound** what actually happened so
the **next** model token is grounded in **your** truth, which is what operators mean by a trustworthy
agent response.

In [1]:
import asyncio
import pathlib
import sys
import traceback

_root = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(_root))
_contracts_src = _root / "packages" / "eXo_adapters" / "packages" / "exo-brain-core-contracts" / "src"
if _contracts_src.is_dir():
    sys.path.insert(0, str(_contracts_src))

try:
    from dotenv import load_dotenv
    load_dotenv(_root / ".env", override=False)
except ImportError:
    pass

print("repo root:", _root)
if not _contracts_src.is_dir():
    print("warn: vendored contracts src missing:", _contracts_src)

repo root: /home/razvansavin/Projects/eXo-brain


## Part 1 — Risk gate knobs (`RiskGateConfig`)

**Story.** Before any runtime adapter runs, product policy usually includes **tier and tool rules**:
which tiers must never execute unattended, which tools always need review, and whether **any**
state-changing call should escalate. `RiskGateConfig` is the declarative bundle for those rules.

**Your task.** Edit **`USER_RISK`** (string tier names and exact tool names), then run the code cell.

**Reading stdout.** This cell only confirms the config object was built. **Part 2** prints one line per
synthetic intent: `decision` (`allow` / `deny` / `escalate`), `reason_code`, `review_required`,
`enforced_mode`.

- `deny_risk_tiers` / `escalate_risk_tiers`: e.g. `"high"`, `"critical"`.
- `deny_tools` / `escalate_tools`: exact registry tool names.
- `escalate_state_changing`: when true, any `is_state_changing=True` intent escalates.

In [2]:
from src.policies.middleware import DeterministicFirstPolicyMiddleware
from src.policies.risk_gates import RiskGateConfig
from src.schemas.tool_io import PolicyAction, RiskTier, ToolCallContext

# ── edit below ─────────────────────────────────────────────────────────────
USER_RISK = {
    "deny_risk_tiers": [],           # e.g. ["critical"]
    "escalate_risk_tiers": ["high"], # demo: HIGH -> ESCALATE
    "deny_tools": [],
    "escalate_tools": [],
    "escalate_state_changing": False,
    "review_channel": "notebook-review",
}


def _tiers(keys: list[str]) -> set[RiskTier]:
    out: set[RiskTier] = set()
    for k in keys:
        try:
            out.add(RiskTier(str(k)))
        except ValueError:
            print("skip unknown RiskTier:", k)
    return out


risk_cfg = RiskGateConfig(
    deny_risk_tiers=_tiers(USER_RISK["deny_risk_tiers"]),
    escalate_risk_tiers=_tiers(USER_RISK["escalate_risk_tiers"]),
    deny_tools=set(USER_RISK["deny_tools"]),
    escalate_tools=set(USER_RISK["escalate_tools"]),
    escalate_state_changing=bool(USER_RISK["escalate_state_changing"]),
    review_channel=str(USER_RISK["review_channel"]),
)
policy_risk = DeterministicFirstPolicyMiddleware(risk_gate_config=risk_cfg)
print("RiskGateConfig ready:", USER_RISK)

RiskGateConfig ready: {'deny_risk_tiers': [], 'escalate_risk_tiers': ['high'], 'deny_tools': [], 'escalate_tools': [], 'escalate_state_changing': False, 'review_channel': 'notebook-review'}


## Part 2 — Probe `before_tool_call` (synthetic tool intents)

**Story.** `PolicyMiddleware.before_tool_call` is the **same** function the orchestrator invokes when
a runtime adapter emits a **tool intent**. This block lets you experiment **without** a model: each
scenario is a hand-built `ToolCallContext`.

**Your task.** Edit **`SCENARIOS`** (tool name, risk tier string, state-changing flag). Re-run.

**Reading stdout.** Each scenario prints **twice**: first with your **Part 1** rules (`policy_risk`),
then with a **relaxed** risk config (`policy_permissive`) so you can see the same tool intent **with**
and **without** tier escalation. For the defaults, **s2** (`delete_row`, HIGH, state-changing) should
move from **escalate** → **allow** (still **deterministic** at execution time — see Part 5).

**Contrast you should internalize:** governance is not “off vs on” — it is **which rules fire** for the
same call shape.

In [3]:
SCENARIOS = [
    {"call_id": "s1", "tool": "read_db", "risk": "low", "state": False},
    {"call_id": "s2", "tool": "delete_row", "risk": "high", "state": True},
    {"call_id": "s3", "tool": "admin_reset", "risk": "medium", "state": True},
]


def _ctx(entry: dict) -> ToolCallContext:
    return ToolCallContext(
        schema_version="1.0",
        call_id=entry["call_id"],
        session_id="nb_sess",
        run_id="nb_run",
        job_id="nb_job",
        task_id="nb_task",
        agent_id="nb_agent",
        provider_id="demo",
        tool_name=entry["tool"],
        arguments={},
        tenant_id="tenant_nb",
        risk_tier=RiskTier(str(entry["risk"])),
        is_state_changing=bool(entry["state"]),
    )


policy_permissive = DeterministicFirstPolicyMiddleware(risk_gate_config=RiskGateConfig())

print("--- with YOUR Part 1 rules (USER_RISK) ---")
for row in SCENARIOS:
    d = policy_risk.before_tool_call(_ctx(row))
    print(
        row["call_id"],
        d.decision.value,
        d.reason_code,
        "review=" + str(d.review_required),
        "enforced_mode=" + str(d.enforced_mode),
    )

print("\n--- contrast: relaxed risk gates (empty RiskGateConfig, same SCENARIOS) ---")
for row in SCENARIOS:
    d = policy_permissive.before_tool_call(_ctx(row))
    print(
        row["call_id"],
        d.decision.value,
        d.reason_code,
        "review=" + str(d.review_required),
        "enforced_mode=" + str(d.enforced_mode),
    )

--- with YOUR Part 1 rules (USER_RISK) ---
s1 allow LOW_RISK_ALLOWED review=False enforced_mode=None
s2 escalate RISK_TIER_REQUIRES_REVIEW review=True enforced_mode=ToolExecutionMode.DETERMINISTIC
s3 allow RISK_WRITE_REQUIRES_DETERMINISTIC review=False enforced_mode=ToolExecutionMode.DETERMINISTIC

--- contrast: relaxed risk gates (empty RiskGateConfig, same SCENARIOS) ---
s1 allow LOW_RISK_ALLOWED review=False enforced_mode=None
s2 allow RISK_WRITE_REQUIRES_DETERMINISTIC review=False enforced_mode=ToolExecutionMode.DETERMINISTIC
s3 allow RISK_WRITE_REQUIRES_DETERMINISTIC review=False enforced_mode=ToolExecutionMode.DETERMINISTIC


## Part 3 — Tenant policy overlay (same risk engine, per-tenant)

**Story.** Global defaults rarely survive multi-tenant reality. `TenantPolicyOverlayStore` merges
**per-tenant** overlay keys onto the same risk gate engine — think “this customer blocks `admin_reset`
even if global policy only escalates HIGH.”

**Your task.** Edit **`USER_OVERLAY`** for tenant `tenant_nb`, then run. Keys mirror overlay fields read
by `RiskGatePolicy` (see `src/policies/risk_gates.py`).

**Reading stdout.** The cell prints **two** decisions for the **same** `ToolCallContext`: first
**without** a tenant overlay on policy (global risk rules only), then **with** `tenant_nb` overlay
(`admin_reset` on the deny list). Beginners should see `allow` flip to **`deny`** only when the overlay
is applied — that is what “per-tenant guard rail” means in code.

**DENY vs ESCALATE (same cell):** the second block probes a **HIGH** risk, state-changing intent. With
**Part 1** defaults (`escalate_risk_tiers` includes **high**), policy returns **`escalate`** — *review
queue semantics*, not a hard block. Compare that feeling to **`deny`** on `admin_reset` above.

**Try this:** remove `"admin_reset"` from `deny_tools`, re-run, and watch the second line follow the first.

In [4]:
from src.tenancy.policy_overlay import TenantPolicyOverlayStore
from src.schemas.tool_io import RiskTier, ToolCallContext

USER_OVERLAY = {
    "deny_tools": ["admin_reset"],
    "escalate_state_changing": False,
    "review_channel": "tenant-security",
}

overlays = TenantPolicyOverlayStore()
overlays.set_overlay("tenant_nb", USER_OVERLAY)
policy_overlay = DeterministicFirstPolicyMiddleware(
    risk_gate_config=risk_cfg,
    tenant_policy_overlays=overlays,
)

probe = ToolCallContext(
    schema_version="1.0",
    call_id="ov1",
    session_id="nb_sess",
    run_id="nb_run",
    job_id="nb_job",
    task_id="nb_task",
    agent_id="nb_agent",
    provider_id="demo",
    tool_name="admin_reset",
    arguments={},
    tenant_id="tenant_nb",
    risk_tier=RiskTier.MEDIUM,
    is_state_changing=False,
)
policy_global_only = DeterministicFirstPolicyMiddleware(risk_gate_config=risk_cfg)
dec_global = policy_global_only.before_tool_call(probe)
print("without tenant overlay (global risk only):", dec_global.decision.value, dec_global.reason_code)

dec = policy_overlay.before_tool_call(probe)
print("with tenant_nb overlay (USER_OVERLAY):    ", dec.decision.value, dec.reason_code, dec.message[:120])

probe_escalate = ToolCallContext(
    schema_version="1.0",
    call_id="ov_esc",
    session_id="nb_sess",
    run_id="nb_run",
    job_id="nb_job",
    task_id="nb_task",
    agent_id="nb_agent",
    provider_id="demo",
    tool_name="delete_row",
    arguments={},
    tenant_id="tenant_nb",
    risk_tier=RiskTier.HIGH,
    is_state_changing=True,
)
esc = policy_overlay.before_tool_call(probe_escalate)
print(
    "HIGH+state delete_row (Part 1 escalate_risk_tiers):",
    esc.decision.value,
    esc.reason_code,
    "review_required=" + str(getattr(esc, "review_required", False)),
)

without tenant overlay (global risk only): allow LOW_RISK_ALLOWED
with tenant_nb overlay (USER_OVERLAY):     deny TOOL_DENIED Tool is blocked by policy configuration.
HIGH+state delete_row (Part 1 escalate_risk_tiers): escalate RISK_TIER_REQUIRES_REVIEW review_required=True


## Part 4 — Deterministic tools you define (handlers + policy + executor)

**Story.** The **deterministic tool runtime** is the contract boundary: the model never executes your
handler. `DeterministicToolExecutor` validates, applies policy again (defense in depth), runs **`fn`**
in-process, and returns a **`ToolResult`**. That is the path you rely on for **money-moving**,
**data-changing**, or **high-risk** operations.

**Your task.** Edit **`USER_TOOLS`**: each item is `{"name", "risk", "state", "fn"}` with **`fn`** a
plain Python callable. Optional keys: **`description`**, **`parameters_schema`** (JSON Schema for the
OpenAI tool surface — used when you go live in Part 8). Re-run `run_tool(...)` at the bottom or add
your own.

**Reading stdout.** `before:` shows policy on the intent. `execute status:` shows executor reality
(`success` vs `blocked`). `mode_used` echoes which execution mode was recorded on the envelope — in
this notebook it stays **`deterministic`** whenever policy blocks or the call is high-impact.

**`calculate_result` (Tutorial 02 parity):** same handler shape as **`tutorial_02_openai_adapter`** —
`operation` (`add` / `subtract` / `multiply` / `divide`) plus **`operand1`** / **`operand2`**. Here it
is registered only in **`ToolRegistry`** (no `@function_tool` in this cell): **`OpenAIAgentsRuntimeAdapter`**
builds SDK tools from the registry (**`build_agent_tools`**), so you see one way production wiring reuses
the same contract as the adapter tutorial.

**`safe_add_proven` (enterprise proof tool):** registered as **MEDIUM + state-changing** so production-style
orchestration prefers the **deterministic executor** (same trust boundary as audited financial tools).
The model supplies **`a`** and **`b`** only; the handler adds **`random_operand`** from per-kernel
**`NB_FORMULA_SECRET`** (never in the user prompt). **`sum = a + b + random_operand`**.

**Acceptance criteria (Part 4 stdout — your kernel’s numbers will differ):**

| Check | Pass signal | Fail signal |
|-------|-------------|-------------|
| Hidden addend | `random_operand` printed (e.g. **4746**) | Only **a+b** appears |
| Governed sum | JSON **`sum`** = a+b+random (e.g. **4751** for a=2,b=3) | **`sum`** equals plain **5** |
| Proof | **`proof_token`** matches **`NB_FORMULA_SECRET`** | Missing or invented token |
| Path | `run_tool` → **`mode_used: DETERMINISTIC`** | Direct `safe_add_proven(...)` only (no policy shell) |

**Part 8 §3 (optional, API key):** replays **11+33** live and prints **`[PASS]` / `[FAIL]`** lines — pass
requires **`safe_add_proven` completed** on the orchestrator path *then* **your** kernel **`sum`** and
**`proof_token`** in the reply (not **44** from mental math or parroting the operator baseline).

**Structured errors:** one **`calculate_result`** call **divides by zero** so you see a deterministic
**`ToolResult`** in **`error`** shape (handler raises; executor wraps — same story as Tutorial 02’s
division demo).

**Contrast at the bottom of the cell:** you will also see **`safe_add` called as plain Python** (no
policy, no metrics). That number is *not* what a production agent path would use — it only shows what
“no governance shell” looks like next to the **same** operation through **`run_tool`**.

In [5]:
import secrets

from src.observability.metrics import RuntimeMetrics
from src.tools.executor import DeterministicToolExecutor
from src.tools.registry import ToolDescriptor, ToolRegistry


def safe_add(a: int, b: int) -> int:
    return a + b


def risky_echo(msg: str) -> str:
    return msg.upper()


def admin_reset() -> str:
    # Demo handler; tenant overlay denies this tool name before it runs in governed paths.
    return "admin-reset-handler-ran"


# Unpredictable per kernel: third addend + proof_token — not visible in the user prompt.
NB_FORMULA_SECRET = secrets.token_hex(8)


def _nb_random_operand() -> int:
    """Stable random addend for this kernel (100..8999); only the handler knows it."""
    return 100 + (int(NB_FORMULA_SECRET[:8], 16) % 8900)


def safe_add_proven(a: int, b: int) -> dict[str, object]:
    a_i, b_i = int(a), int(b)
    random_operand = _nb_random_operand()
    total = a_i + b_i + random_operand
    return {
        "operand_a": a_i,
        "operand_b": b_i,
        "random_operand": random_operand,
        "sum": total,
        "proof_token": NB_FORMULA_SECRET,
        "formula": f"{a_i}+{b_i}+{random_operand}=={total}",
    }


def _nb_calculate_result(operation: str, operand1: float, operand2: float) -> dict[str, object]:
    """Same arithmetic contract as tutorial_02 (registry handler; policy + executor wrap it here)."""
    op = str(operation).strip().lower()
    if op == "add":
        value = float(operand1) + float(operand2)
    elif op == "subtract":
        value = float(operand1) - float(operand2)
    elif op == "multiply":
        value = float(operand1) * float(operand2)
    elif op == "divide":
        if float(operand2) == 0:
            raise ValueError("division by zero is not allowed")
        value = float(operand1) / float(operand2)
    else:
        raise ValueError(f"unknown operation: {operation!r}")
    return {
        "operation": op,
        "operand1": float(operand1),
        "operand2": float(operand2),
        "result": value,
    }


_CALCULATE_RESULT_SCHEMA: dict[str, object] = {
    "type": "object",
    "properties": {
        "operation": {
            "type": "string",
            "description": "One of: add, subtract, multiply, divide",
        },
        "operand1": {"type": "number"},
        "operand2": {"type": "number"},
    },
    "required": ["operation", "operand1", "operand2"],
}

_SAFE_ADD_PROVEN_SCHEMA: dict[str, object] = {
    "type": "object",
    "properties": {
        "a": {"type": "integer", "description": "First addend (visible to the model)."},
        "b": {"type": "integer", "description": "Second addend (visible to the model)."},
    },
    "required": ["a", "b"],
}


def _nb_print_proof_reference(a: int, b: int, *, title: str) -> tuple[int, int]:
    """Print kernel-only operands for demos; returns (random_operand, governed_sum)."""
    r = _nb_random_operand()
    governed = a + b + r
    plain = a + b
    print(title)
    print(f"  random_operand (handler-only): {r}")
    print(f"  governed sum {a}+{b}+{r} => {governed}  |  plain {a}+{b} => {plain} (wrong without tool)")
    print(f"  proof_token (this kernel): {NB_FORMULA_SECRET}")
    return r, governed


USER_TOOLS = [
    {"name": "safe_add", "risk": RiskTier.LOW, "state": False, "fn": safe_add},
    {
        "name": "safe_add_proven",
        "risk": RiskTier.MEDIUM,
        "state": True,
        "fn": safe_add_proven,
        "description": (
            "Adds a and b plus a hidden per-tenant random_operand; returns sum, formula, and proof_token."
        ),
        "parameters_schema": _SAFE_ADD_PROVEN_SCHEMA,
    },
    {
        "name": "calculate_result",
        "risk": RiskTier.LOW,
        "state": False,
        "fn": _nb_calculate_result,
        "description": "Basic arithmetic: add, subtract, multiply, or divide two operands.",
        "parameters_schema": _CALCULATE_RESULT_SCHEMA,
    },
    {"name": "risky_echo", "risk": RiskTier.HIGH, "state": True, "fn": risky_echo},
    {"name": "admin_reset", "risk": RiskTier.MEDIUM, "state": True, "fn": admin_reset},
]

registry = ToolRegistry()
for spec in USER_TOOLS:
    registry.register(
        ToolDescriptor(
            name=spec["name"],
            handler=spec["fn"],
            risk_tier=spec["risk"],
            is_state_changing=spec["state"],
            description=str(spec.get("description", "")),
            parameters_schema=dict(spec["parameters_schema"]) if spec.get("parameters_schema") else {},
        )
    )

metrics = RuntimeMetrics()
executor = DeterministicToolExecutor(
    registry=registry,
    policy=policy_overlay,
    metrics=metrics,
)


def run_tool(
    name: str,
    args: dict,
    call_id: str,
    *,
    risk_tier: RiskTier = RiskTier.LOW,
    is_state_changing: bool = False,
) -> None:
    call = ToolCallContext(
        schema_version="1.0",
        call_id=call_id,
        session_id="nb_sess",
        run_id="nb_run",
        job_id="nb_job",
        task_id="nb_task",
        agent_id="nb_agent",
        provider_id="demo",
        tool_name=name,
        arguments=args,
        tenant_id="tenant_nb",
        risk_tier=risk_tier,
        is_state_changing=is_state_changing,
    )
    try:
        pre = policy_overlay.before_tool_call(call)
        print("before:", pre.decision.value, pre.reason_code)
        out = executor.execute(call)
        print("execute status:", out.status.value)
        err = out.error
        err_code = getattr(err, "code", None) if err is not None else None
        err_msg = getattr(err, "message", "") if err is not None else ""
        if err_msg is None:
            err_msg = ""
        print("  error:", err_code, str(err_msg)[:200])
        print("  mode_used:", out.execution.mode_used)
    except Exception:
        traceback.print_exc()


run_tool("safe_add", {"a": 2, "b": 3}, "tc_add", risk_tier=RiskTier.LOW, is_state_changing=False)
_demo_r, _demo_sum = _nb_print_proof_reference(
    2,
    3,
    title="-- safe_add_proven: enterprise proof (sum = a + b + kernel random_operand) --",
)
run_tool(
    "safe_add_proven",
    {"a": 2, "b": 3},
    "tc_prov",
    risk_tier=RiskTier.MEDIUM,
    is_state_changing=True,
)
_proven_payload = safe_add_proven(2, 3)
print("  safe_add_proven JSON:", _proven_payload)
if _proven_payload.get("sum") == _demo_sum and _proven_payload.get("proof_token") == NB_FORMULA_SECRET:
    print("  [PASS] Part 4 local proof — sum and proof_token match kernel baseline")
else:
    print("  [FAIL] Part 4 local proof — JSON does not match kernel baseline (unexpected)")
print("  → Part 8 §3 will require the live model to cite this sum and proof_token (not plain 5).")
run_tool(
    "calculate_result",
    {"operation": "multiply", "operand1": 8, "operand2": 9},
    "tc_calc",
    risk_tier=RiskTier.LOW,
    is_state_changing=False,
)
print("-- calculate_result divide-by-zero → structured TOOL_EXECUTION_ERROR --")
run_tool(
    "calculate_result",
    {"operation": "divide", "operand1": 10, "operand2": 0},
    "tc_div0",
    risk_tier=RiskTier.LOW,
    is_state_changing=False,
)
run_tool("risky_echo", {"msg": "hello"}, "tc_echo", risk_tier=RiskTier.HIGH, is_state_changing=True)
run_tool("admin_reset", {}, "tc_denied", risk_tier=RiskTier.MEDIUM, is_state_changing=False)
print("metrics counters:", metrics.counters)
print("NB_FORMULA_SECRET for this kernel (compare to live model reply in Part 8):", NB_FORMULA_SECRET)

print()
print("Contrast — same math, no policy / no executor / no metrics (not a supported agent path):")
print("  safe_add(2, 3) =>", safe_add(2, 3))
print("(Above, run_tool('safe_add', ...) went through policy + DeterministicToolExecutor + metrics.)")

before: allow LOW_RISK_ALLOWED
execute status: success
  error: None 
  mode_used: ToolExecutionMode.DETERMINISTIC
-- safe_add_proven: enterprise proof (sum = a + b + kernel random_operand) --
  random_operand (handler-only): 5706
  governed sum 2+3+5706 => 5711  |  plain 2+3 => 5 (wrong without tool)
  proof_token (this kernel): 5bd266b261299aa7
before: allow RISK_WRITE_REQUIRES_DETERMINISTIC
execute status: success
  error: None 
  mode_used: ToolExecutionMode.DETERMINISTIC
  safe_add_proven JSON: {'operand_a': 2, 'operand_b': 3, 'random_operand': 5706, 'sum': 5711, 'proof_token': '5bd266b261299aa7', 'formula': '2+3+5706==5711'}
  [PASS] Part 4 local proof — sum and proof_token match kernel baseline
  → Part 8 §3 will require the live model to cite this sum and proof_token (not plain 5).
before: allow LOW_RISK_ALLOWED
execute status: success
  error: None 
  mode_used: ToolExecutionMode.DETERMINISTIC
-- calculate_result divide-by-zero → structured TOOL_EXECUTION_ERROR --
before: allo

## Part 5 — `select_execution_mode` (capability + policy)

**Story.** Even when policy **allows** a call, the product still chooses **how** it runs. Capability maps
describe the adapter (reliability, structured output support, etc.). `select_execution_mode` merges
**policy** (`enforced_mode`, risk tier, state-changing) with **capability** to pick
`deterministic` vs `provider_native`.

**Your task.** Edit **`CAPABILITY_VARIANTS`**: each entry’s `"kwargs"` is passed to `ProviderCapabilityMap`.
Compare the printed modes for the **same** `PolicyDecision.ALLOW` but different synthetic tool calls.

**Reading stdout.** **HIGH + state-changing** should stay **`deterministic`** even when the “weak” map
would otherwise prefer the provider — safety wins. **LOW** calls may show **`provider_native`** when
capabilities look “healthy”; that is the fast path, not a bypass for writes you care about.

**Contrast to watch:** for the **same** `low` tool call, `weak_capabilities` vs `strong_capabilities`
often prints **different** modes (`provider_native` vs `deterministic`) because `should_force_deterministic`
kicks in when the map looks “weak”. That is the framework nudging you toward safer execution without
changing your tool code.

In [6]:
from src.runtime.capability_map import ProviderCapabilityMap
from src.runtime.mode_selector import select_execution_mode
from src.schemas.tool_io import PolicyAction, PolicyDecision, RiskTier, ToolCallContext, ToolExecutionMode

CAPABILITY_VARIANTS = [
    {"label": "weak_capabilities", "kwargs": {"provider_id": "demo", "reliability_score": 5}},
    {
        "label": "strong_capabilities",
        "kwargs": {
            "provider_id": "demo",
            "supports_function_calling": True,
            "supports_structured_output": True,
            "reliability_score": 5,
        },
    },
]

allow = PolicyDecision(
    schema_version="1.0",
    decision=PolicyAction.ALLOW,
    reason_code="LOW_RISK_ALLOWED",
    message="ok",
    enforced_mode=None,
)

low = ToolCallContext(
    schema_version="1.0",
    call_id="m1",
    session_id="nb_sess",
    run_id="nb_run",
    job_id="nb_job",
    task_id="nb_task",
    agent_id="nb_agent",
    provider_id="demo",
    tool_name="safe_add",
    arguments={},
    tenant_id="tenant_nb",
    risk_tier=RiskTier.LOW,
    is_state_changing=False,
)
high = ToolCallContext(
    schema_version="1.0",
    call_id="m2",
    session_id="nb_sess",
    run_id="nb_run",
    job_id="nb_job",
    task_id="nb_task",
    agent_id="nb_agent",
    provider_id="demo",
    tool_name="risky_echo",
    arguments={"msg": "x"},
    tenant_id="tenant_nb",
    risk_tier=RiskTier.HIGH,
    is_state_changing=True,
)

print("Same tool intents; only the capability map changes:\n")
for variant in CAPABILITY_VARIANTS:
    caps = ProviderCapabilityMap(**variant["kwargs"])
    print(variant["label"], "low ->", select_execution_mode(low, caps, allow).value)
    print(variant["label"], "high ->", select_execution_mode(high, caps, allow).value)

Same tool intents; only the capability map changes:

weak_capabilities low -> provider_native
weak_capabilities high -> deterministic
strong_capabilities low -> provider_native
strong_capabilities high -> deterministic


## Part 6 — Ingress gate chain (pre-model guard rails)

**Story.** **Ingress** answers: “Should this *text* become a billable model turn?” It runs **before**
the orchestrator. Custom rules, classifiers, and profile defaults all collapse into an ordered gate
chain with explicit **`gate_id`** and **`reason_code`** — ideal for SOC-style reviews.

**Your task.** Edit **`INGRESS_OVERLAY`** (profile, classifier mode, custom rules). Use
**`evaluate_prompt`** to send benign vs sensitive sample strings.

**Reading stdout.** The code runs **two** prompts back-to-back: a **benign** string (should **allow**)
and a **sensitive** string containing `SECRET_KEY` (should **deny** with your custom rule). That is the
simplest **with vs without** story for ingress: same chain, different user text, opposite outcomes —
and **no** model spend on the denied line.

**Part 8 reuse.** The same `chain` object is reused when an API key is present so you can show a live
turn **blocked at ingress** vs **allowed through to the model**.

In [7]:
from src.policies.ingress_gates import (
    IngressDecision,
    IngressGateChain,
    IngressTurnContext,
    build_ingress_gate_chain_from_overlay,
)
from src.policies.ingress_profiles import resolve_ingress_profile_settings
from src.schemas.tool_io import PolicyAction


def evaluate_prompt(
    chain: IngressGateChain,
    prompt: str,
    session_id: str = "nb-ingress",
) -> IngressDecision:
    ctx = IngressTurnContext(
        tenant_id="tenant_nb",
        session_id=session_id,
        correlation_id="corr-" + session_id,
        transport="notebook",
        user_input=prompt,
    )
    decision = chain.evaluate(ctx)
    msg = decision.message or ""
    print(decision.decision.value, decision.gate_id, decision.reason_code, "|", msg[:100])
    return decision


INGRESS_OVERLAY = {
    "ingress_profile": "baseline",
    "ingress_classifier_mode": "off",
    "ingress_custom_rules": [
        {
            "rule_id": "nb-block-secret",
            "action": "deny",
            "match_type": "contains_any",
            "patterns": ["SECRET_KEY", "BEGIN PRIVATE KEY"],
            "reason_code": "NB_SECRET_PATTERN",
            "message": "Blocked in notebook demo.",
        },
    ],
}

res = resolve_ingress_profile_settings(INGRESS_OVERLAY)
print("resolved profile:", res.profile_name, "custom rules:", len(res.custom_rules))

chain = build_ingress_gate_chain_from_overlay(INGRESS_OVERLAY)
evaluate_prompt(chain, "hello world")
d = evaluate_prompt(chain, "paste SECRET_KEY=abc here")
assert d.decision == PolicyAction.DENY
print("PASS ingress deny on secret pattern")

resolved profile: baseline custom rules: 1
allow ingress-gate-chain INGRESS_ALLOW_DEFAULT | Turn allowed by ingress gate chain.
deny ingress-custom-rules NB_SECRET_PATTERN | Blocked in notebook demo.
PASS ingress deny on secret pattern


## Checkpoint — before Part 7 (orchestrator)

If anything below **fails**, use **Run All Above** from the next cell, or restart the kernel and run
from the **bootstrap** cell through **Part 6** without skipping.

**You should have seen:** Part 2 **escalate** on `s2`, Part 3 **`deny`** on `admin_reset` with overlay,
Part 4 **`success`** and one **`blocked`/`error`** line for divide-by-zero, Part 6 **`PASS ingress deny`**.

In [8]:
_missing = []
for _name in (
    "policy_overlay",
    "registry",
    "executor",
    "metrics",
    "chain",
    "evaluate_prompt",
    "NB_FORMULA_SECRET",
    "risk_cfg",
):
    if _name not in globals():
        _missing.append(_name)
if _missing:
    raise RuntimeError(
        "Checkpoint failed — re-run notebook from bootstrap through Part 6. Missing: " + ", ".join(_missing)
    )
print("CHECKPOINT OK — continue to Part 7 (stub orchestrator) and optional Part 8 (live API).")

CHECKPOINT OK — continue to Part 7 (stub orchestrator) and optional Part 8 (live API).


## Part 7 — One-turn orchestrator (stub stream; no API key)

**Story.** The **orchestrator** ties the runtime adapter, policy, and executor into one **async event
stream** (`tool_progress`, `tool_intent`, `output_delta`, `run_complete`). For tests and notebooks,
`OpenAIAgentsRuntimeAdapter` supports **`planned_tool_call`**: a synthetic tool intent without calling
OpenAI. That lets you see **policy + deterministic execution + submit_tool_results** end-to-end.

**Your task.** Edit **`planned_tool_call`** (`tool_name`, `arguments`, `risk_tier`, `is_state_changing`)
so it matches a **registered** tool from Part 4. The default uses **`calculate_result`** × **8×9** with
**`risk_tier: medium`** and **`is_state_changing: true`** so you see **queued → running → completed**
without an API key.

**Important with your Part 1 config:** `USER_RISK` sets **`escalate_risk_tiers: ["high"]`**. A synthetic
intent marked **`high`** is **escalated → blocked** (`POLICY_BLOCKED`) — same as Part 2’s `s2` line. That is
correct policy behaviour, not a broken orchestrator. The cell runs a **second** planned call with
**`high`** to show that contrast after the completed **medium** path.

**Reading stdout (first run).** **queued → running → completed**, then **`output_delta`** /
**`run_complete`**. **Second run:** **queued → failed** with **`POLICY_BLOCKED`** — ties Part 2 to the stream.

**With vs without OpenAI:** this part is **without** billing — `planned_tool_call` injects a tool
intent. **Part 8** (optional, API key) reuses the same **`planned_tool_call`** mechanism for **§2–§4**
governed proofs, plus ingress and raw Agents SDK contrasts; **8.5** (off by default) is model-driven
diagnostic only.

In [9]:
from src.core.orchestrator import Orchestrator
from src.runtime.openai_agents_runtime import OpenAIAgentsRuntimeAdapter
from src.schemas.events import RuntimeEventType

orch = Orchestrator(
    runtime_adapter=OpenAIAgentsRuntimeAdapter(),
    policy_middleware=policy_overlay,
    tool_executor=DeterministicToolExecutor(registry=registry, policy=policy_overlay, metrics=metrics),
)

ctx = {
    "run_id": "nb_orch_run",
    "job_id": "nb_orch_job",
    "task_id": "nb_orch_task",
    "agent_id": "nb_orch_agent",
    "planned_tool_call": {
        "call_id": "tc_orch_1",
        "tool_name": "calculate_result",
        "arguments": {"operation": "multiply", "operand1": 8, "operand2": 9},
        "risk_tier": "medium",
        "is_state_changing": True,
    },
}


def _progress_states(event_pairs: list) -> list[str]:
    states: list[str] = []
    for etype, payload in event_pairs:
        if etype == RuntimeEventType.TOOL_PROGRESS.value and isinstance(payload, dict):
            st = payload.get("state")
            if isinstance(st, str):
                states.append(st)
    return states


async def _run_planned(ctx: dict, *, label: str) -> list:
    print(f"\n-- {label} --")
    out: list = []
    async for ev in orch.run_turn("sess_nb", "run tool", ctx):
        out.append((ev.event_type.value, ev.payload))
        print(ev.event_type.value, ev.payload)
    return out


try:
    loop = asyncio.get_running_loop()
except RuntimeError:
    events_ok = asyncio.run(_run_planned(ctx, label="MEDIUM + state-changing (expect completed)"))
else:
    try:
        import nest_asyncio
        nest_asyncio.apply()
        events_ok = loop.run_until_complete(_run_planned(ctx, label="MEDIUM + state-changing (expect completed)"))
    except ImportError:
        print("Install nest-asyncio for Jupyter: pip install nest-asyncio")
        raise

states_ok = _progress_states(events_ok)
assert RuntimeEventType.RUN_COMPLETE.value in [t for t, _ in events_ok]
assert "completed" in states_ok, f"expected completed tool_progress, got states={states_ok!r}"
print("PASS orchestrator stream — deterministic completed path")

ctx_high = dict(ctx)
ctx_high["planned_tool_call"] = {
    **ctx["planned_tool_call"],
    "call_id": "tc_orch_high_blocked",
    "risk_tier": "high",
}

try:
    loop = asyncio.get_running_loop()
except RuntimeError:
    events_blk = asyncio.run(_run_planned(ctx_high, label="HIGH risk (expect POLICY_BLOCKED — Part 1 escalate)"))
else:
    import nest_asyncio
    nest_asyncio.apply()
    events_blk = loop.run_until_complete(
        _run_planned(ctx_high, label="HIGH risk (expect POLICY_BLOCKED — Part 1 escalate)")
    )

states_blk = _progress_states(events_blk)
assert "failed" in states_blk, f"expected failed tool_progress for HIGH, got {states_blk!r}"
print("PASS orchestrator stream — HIGH intent blocked by policy (consistent with Part 2)")


-- MEDIUM + state-changing (expect completed) --
tool_progress {'call_id': 'tc_orch_1', 'tool_name': 'calculate_result', 'state': 'queued', 'tool_status': '', 'error_code': '', 'job_id': '', 'lease_token': '', 'lease_expires_at_epoch': '', 'claim_attempt': ''}
tool_progress {'call_id': 'tc_orch_1', 'tool_name': 'calculate_result', 'state': 'running', 'tool_status': '', 'error_code': '', 'job_id': '', 'lease_token': '', 'lease_expires_at_epoch': '', 'claim_attempt': ''}
tool_progress {'call_id': 'tc_orch_1', 'tool_name': 'calculate_result', 'state': 'completed', 'tool_status': 'success', 'error_code': 'None', 'job_id': '', 'lease_token': '', 'lease_expires_at_epoch': '', 'claim_attempt': ''}
output_delta {'text': "- calculate_result (tc_orch_1): success → {'operation': 'multiply', 'operand1': 8.0, 'operand2': 9.0, 'result': 72.0}"}
run_complete {'status': 'completed', 'tool_results_count': 1, 'tool_results_summary': "- calculate_result (tc_orch_1): success → {'operation': 'multiply', '

## Part 8 — Optional live contrasts (`OPENAI_API_KEY`)

**Story.** Parts **1–7** are fully local. Part **8** is optional: real ingress, governed orchestrator
streams with **`planned_tool_call`** (same as Part 7), and **raw SDK** anti-patterns for contrast.

**Prerequisites:** Parts **1–6** (`policy_overlay`, `registry`, `executor`, `chain`). Part **4** registers
`admin_reset`, `safe_add_proven`, `calculate_result`.

**Skip without a key:** Run **8.1** once; later cells print skip if `OPENAI_API_KEY` is unset (CI-safe).

**Cost flags:** `NB_LIVE_INGRESS`, `NB_LIVE_POLICY`, `NB_LIVE_MATH`, `NB_LIVE_CALC` (default on);
`NB_LIVE_RAW_CALC_CONTRAST`, `NB_LIVE_MODEL_DRIVEN` (default off). Set any to `0` / `false` / `off` to skip.

### Part 8.0 — Bridge from Part 7

| Part 7 (no API) | Part 8 (optional API) |
|-----------------|------------------------|
| `planned_tool_call` → `tool_intent` / `tool_progress` | Same orchestrator mechanism |
| Stub / local stream | Plus ingress evaluate + raw Agents SDK contrast |
| Proves policy + executor | §2–§4 **PASS** when `planned_tool_call` is injected (reliable) |

**Not the same as “ask the model nicely”.** OpenAI **delegating** tools (`FunctionTool` wrappers) often run
handlers without emitting `TOOL_INTENT` on the orchestrator stream. Part **8.6** (optional) explores that;
**§2–§4 verification** uses **`planned_tool_call`** only.

In [10]:
import asyncio
import os

HAS_OPENAI_KEY = bool(os.environ.get("OPENAI_API_KEY", "").strip())
print("OPENAI_API_KEY set:", HAS_OPENAI_KEY)

_live_gov_summary: dict[str, str] = {}


def _nb_live_on(name: str, default: str = "1") -> bool:
    raw = os.environ.get(name, default)
    if raw is None:
        return True
    s = str(raw).strip().lower()
    return s not in ("0", "false", "no", "off", "")


if not HAS_OPENAI_KEY:
    print("Part 8 setup — skip live cells until OPENAI_API_KEY is set in .env")
    LIVE_INGRESS = LIVE_POLICY = LIVE_MATH = LIVE_CALC = False
    LIVE_RAW_CALC = LIVE_MODEL_DRIVEN = False
else:
    LIVE_INGRESS = _nb_live_on("NB_LIVE_INGRESS", "1")
    LIVE_POLICY = _nb_live_on("NB_LIVE_POLICY", "1")
    LIVE_MATH = _nb_live_on("NB_LIVE_MATH", "1")
    LIVE_CALC = _nb_live_on("NB_LIVE_CALC", "1")
    LIVE_RAW_CALC = _nb_live_on("NB_LIVE_RAW_CALC_CONTRAST", "0")
    LIVE_MODEL_DRIVEN = _nb_live_on("NB_LIVE_MODEL_DRIVEN", "0")
    print(
        "Live flags:",
        {
            "NB_LIVE_INGRESS": LIVE_INGRESS,
            "NB_LIVE_POLICY": LIVE_POLICY,
            "NB_LIVE_MATH": LIVE_MATH,
            "NB_LIVE_CALC": LIVE_CALC,
            "NB_LIVE_RAW_CALC_CONTRAST": LIVE_RAW_CALC,
            "NB_LIVE_MODEL_DRIVEN": LIVE_MODEL_DRIVEN,
        },
    )

    from agents import Agent, Runner, function_tool
    from agents.items import ItemHelpers, MessageOutputItem, ToolCallItem, ToolCallOutputItem
    from agents.stream_events import RunItemStreamEvent

    from src.core.orchestrator import Orchestrator
    from src.runtime.openai_agents_runtime import OpenAIAgentsRuntimeAdapter
    from src.schemas.events import RuntimeEventType
    from src.schemas.tool_io import PolicyAction

    @function_tool
    def raw_admin_reset() -> str:
        return "UNGOVERNED_RESET_DEMO_RAN"

    @function_tool
    def sloppy_add_proven(a: int, b: int) -> dict[str, object]:
        a_i, b_i = int(a), int(b)
        wrong = a_i + b_i + 999
        return {
            "operand_a": a_i,
            "operand_b": b_i,
            "random_operand": 0,
            "sum": wrong,
            "proof_token": "RAW_UNGOVERNED_STATIC_PROOF",
            "formula": f"{a_i}+{b_i}+buggy_anchor=={wrong}",
        }

    @function_tool
    def raw_calculate_broken(operation: str, operand1: float, operand2: float) -> dict[str, object]:
        op = str(operation).strip().lower()
        o1, o2 = float(operand1), float(operand2)
        bad = o1 * o2 + 1000.0 if op == "multiply" else o1 + o2 + 1000.0
        return {"operation": op, "operand1": o1, "operand2": o2, "result": bad}

    def _check_substring(haystack: str, needle: str, *, label: str) -> None:
        if needle and needle in haystack:
            print("  CHECK OK:", label)
        elif needle:
            print("  CHECK (soft):", label, "- expected substring not in model text.")

    def _nb_verify_line(ok: bool, label: str, *, level: str = "PASS") -> bool:
        tag = level if ok else ("WARN" if level == "WARN" else "FAIL")
        print(f"  [{tag}] {label}")
        return ok

    def _nb_live_math_operands() -> tuple[int, int]:
        def _parse(name: str, default: int) -> int:
            raw = os.environ.get(name, "").strip()
            if not raw:
                return default
            try:
                return int(raw)
            except ValueError:
                print(f"  warn: {name}={raw!r} invalid — using default {default}")
                return default

        return _parse("NB_LIVE_MATH_A", 11), _parse("NB_LIVE_MATH_B", 33)

    async def _raw_sdk_trace(user_input: str, *, tools: list, instructions: str) -> str:
        agent = Agent(name="raw-nb-ungoverned", instructions=instructions, model="gpt-4o-mini", tools=tools)
        result = Runner.run_streamed(agent, user_input)
        parts: list[str] = []
        async for event in result.stream_events():
            if not isinstance(event, RunItemStreamEvent):
                continue
            item = event.item
            if isinstance(item, MessageOutputItem):
                text = ItemHelpers.text_message_output(item)[:500]
                print("  raw:", "message:", text)
                if text.strip():
                    parts.append(text)
            elif isinstance(item, ToolCallItem):
                print("  raw:", "tool_call:", type(item).__name__)
            elif isinstance(item, ToolCallOutputItem):
                out = getattr(item, "output", None)
                print("  raw:", "tool_output:", str(out)[:500])
                if out is not None:
                    parts.append(str(out))
        return " ".join(parts)

    _live_session_meta = {
        "tenant_id": "tenant_nb",
        "agent_id": "notebook-governed-live",
        "instructions": (
            "You are a compact notebook assistant. You MUST use tools when asked — do not refuse. "
            "When the user says to call admin_reset, call admin_reset immediately with no arguments. "
            "When the user asks for safe_add_proven, call it once with integer keys a and b; "
            "then quote only the sum and proof_token fields from the tool JSON (never guess a+b). "
            "When the user asks for calculate_result, call it once with operation, operand1, operand2; "
            "then state the result field from the tool JSON."
        ),
        "model": "gpt-4o-mini",
    }

    live_adapter = OpenAIAgentsRuntimeAdapter(
        provider_id="openai",
        tool_registry=registry,
        tool_executor=executor,
    )
    live_orch = Orchestrator(
        runtime_adapter=live_adapter,
        policy_middleware=policy_overlay,
        tool_executor=executor,
    )

    async def _governed_turn(
        session_id: str,
        user_input: str,
        *,
        run_label: str,
        planned_tool_call: dict[str, object] | None = None,
    ) -> dict[str, object]:
        ctx: dict[str, object] = {
            "run_id": "nb_live_" + run_label,
            "job_id": "nb_live_job",
            "task_id": "nb_live_task",
            "agent_id": "nb_live_agent",
            "session_metadata": dict(_live_session_meta),
        }
        if planned_tool_call is not None:
            ctx["planned_tool_call"] = planned_tool_call
        parts: list[str] = []
        tool_intents: list[str] = []
        tools_completed: list[str] = []
        policy_blocked: list[str] = []
        async for ev in live_orch.run_turn(session_id, user_input, ctx):
            snippet = str(ev.payload)[:260]
            if ev.event_type == RuntimeEventType.TOOL_PROGRESS:
                print("  gov:", ev.event_type.value, snippet)
                if isinstance(ev.payload, dict):
                    tn = ev.payload.get("tool_name")
                    state = ev.payload.get("state")
                    err = ev.payload.get("error_code")
                    if isinstance(tn, str) and tn:
                        if state == "completed":
                            tools_completed.append(tn)
                        if state == "failed" and str(err) == "POLICY_BLOCKED":
                            policy_blocked.append(tn)
            elif ev.event_type == RuntimeEventType.TOOL_INTENT:
                tn = ev.tool_call.tool_name if ev.tool_call else ""
                print("  gov:", ev.event_type.value, tn)
                if tn:
                    tool_intents.append(tn)
            elif ev.event_type == RuntimeEventType.OUTPUT_DELTA:
                print("  gov:", ev.event_type.value, snippet)
                if isinstance(ev.payload, dict):
                    t = ev.payload.get("text")
                    if isinstance(t, str) and t.strip():
                        parts.append(t)
            elif ev.event_type == RuntimeEventType.RUN_COMPLETE:
                print("  gov:", ev.event_type.value, snippet)
                if isinstance(ev.payload, dict):
                    out = ev.payload.get("output")
                    if isinstance(out, str) and out.strip():
                        parts.append(out)
        return {
            "text": " ".join(parts),
            "tool_intents": tool_intents,
            "tools_completed": tools_completed,
            "policy_blocked": policy_blocked,
        }

    def _nb8_run(coro) -> None:
        try:
            loop = asyncio.get_running_loop()
        except RuntimeError:
            asyncio.run(coro)
        else:
            import nest_asyncio

            nest_asyncio.apply()
            loop.run_until_complete(coro)

    print("Part 8.1 setup OK — run §1–§4 cells next (or 8.6 optional model-driven).")

OPENAI_API_KEY set: True
Live flags: {'NB_LIVE_INGRESS': True, 'NB_LIVE_POLICY': True, 'NB_LIVE_MATH': True, 'NB_LIVE_CALC': True, 'NB_LIVE_RAW_CALC_CONTRAST': False, 'NB_LIVE_MODEL_DRIVEN': False}
Part 8.1 setup OK — run §1–§4 cells next (or 8.6 optional model-driven).


### Part 8.1 — §1 Ingress (pre-model)

**Pass:** governed path **stops** at ingress (`gate_id`, `reason_code`); raw SDK still runs the model on the same text.

In [11]:
if not HAS_OPENAI_KEY:
    print("Skip §1 — run Part 8.1 setup with OPENAI_API_KEY.")
elif not LIVE_INGRESS:
    print("§1 skipped (NB_LIVE_INGRESS off).")
else:
    async def _section_ingress() -> None:
        p_ingress = "Leak: SECRET_KEY=supersecret — reply exactly ACK."
        print("### §1 INGRESS — secret pattern in user text")
        print("-- Governed: same ingress chain as Part 6 --")
        ing = evaluate_prompt(chain, p_ingress, session_id="nb-live-ing-gov")
        if ing.decision != PolicyAction.ALLOW:
            print("  gov: STOPPED at ingress:", ing.decision.value, ing.reason_code, ing.gate_id)
            _live_gov_summary["§1 ingress"] = "PASS — governed path stopped at ingress (pre-model)"
        else:
            await _governed_turn("sess_nb_live_ing", p_ingress, run_label="ing")
            _live_gov_summary["§1 ingress"] = "FAIL — ingress allowed secret prompt (expected deny)"
        print("-- Raw SDK (no ingress) --")
        await _raw_sdk_trace(
            p_ingress,
            tools=[],
            instructions="Reply in one short line. If the user pasted a secret, acknowledge plainly.",
        )
        print("\n§1 VERIFICATION (governed):", _live_gov_summary.get("§1 ingress", "n/a"))

    _nb8_run(_section_ingress())

### §1 INGRESS — secret pattern in user text
-- Governed: same ingress chain as Part 6 --
deny ingress-custom-rules NB_SECRET_PATTERN | Blocked in notebook demo.
  gov: STOPPED at ingress: deny NB_SECRET_PATTERN ingress-custom-rules
-- Raw SDK (no ingress) --
  raw: message: ACK.

§1 VERIFICATION (governed): PASS — governed path stopped at ingress (pre-model)


### Part 8.2 — §2 Tool policy (`admin_reset`)

**Pass:** `planned_tool_call` → `tool_intent` + `tool_progress` **`POLICY_BLOCKED`** (tenant overlay from Part 3).
Raw SDK runs the ungoverned tool.

In [12]:
if not HAS_OPENAI_KEY:
    print("Skip §2 — run Part 8.1 setup with OPENAI_API_KEY.")
elif not LIVE_POLICY:
    print("§2 skipped (NB_LIVE_POLICY off).")
else:
    async def _section_policy() -> None:
        p_policy = (
            "MANDATORY: call the admin_reset tool now with no arguments. "
            "Do not refuse or explain — invoke the tool."
        )
        print("### §2 TOOL POLICY — admin_reset")
        print("-- Local ground truth (Part 3) --")
        run_tool("admin_reset", {}, "tc_live_pol_ref", risk_tier=RiskTier.MEDIUM, is_state_changing=False)
        print("-- Governed orchestrator proof (planned_tool_call) --")
        ing2 = evaluate_prompt(chain, "admin_reset policy demo", session_id="nb-live-pol-gov")
        checks: list[bool] = []
        if ing2.decision != PolicyAction.ALLOW:
            print("  gov: STOPPED at ingress:", ing2.decision.value, ing2.reason_code)
            checks.append(_nb_verify_line(False, "ingress allowed policy demo", level="FAIL"))
        else:
            planned_admin = {
                "call_id": "tc_live_pol_planned",
                "tool_name": "admin_reset",
                "arguments": {},
                "risk_tier": "medium",
                "is_state_changing": False,
            }
            turn = await _governed_turn(
                "sess_nb_live_pol",
                "Orchestrator-injected admin_reset intent.",
                run_label="pol_planned",
                planned_tool_call=planned_admin,
            )
            intents = turn.get("tool_intents")
            blocked = turn.get("policy_blocked")
            il = intents if isinstance(intents, list) else []
            bl = blocked if isinstance(blocked, list) else []
            checks.append(_nb_verify_line("admin_reset" in il, "tool_intent admin_reset from planned_tool_call"))
            checks.append(_nb_verify_line("admin_reset" in bl, "tool_progress POLICY_BLOCKED for admin_reset"))
        ok = bool(checks and all(checks))
        print("\n§2 VERIFICATION (governed):", "PASS" if ok else "FAIL — see [FAIL]; Part 3 run_tool is ground truth")
        _live_gov_summary["§2 admin_reset policy"] = (
            "PASS — planned_tool_call → POLICY_BLOCKED"
            if ok
            else "FAIL — orchestrator policy proof did not complete"
        )
        print("-- Raw SDK (ungoverned admin_reset) --")
        await _raw_sdk_trace(
            p_policy,
            tools=[raw_admin_reset],
            instructions="If the user asks for admin_reset, call the admin_reset tool once.",
        )

    _nb8_run(_section_policy())

### §2 TOOL POLICY — admin_reset
-- Local ground truth (Part 3) --
before: deny TOOL_DENIED
execute status: blocked
  error: POLICY_BLOCKED Tool is blocked by policy configuration.
  mode_used: ToolExecutionMode.DETERMINISTIC
-- Governed orchestrator proof (planned_tool_call) --
allow ingress-gate-chain INGRESS_ALLOW_DEFAULT | Turn allowed by ingress gate chain.
  gov: tool_progress {'call_id': 'tc_live_pol_planned', 'tool_name': 'admin_reset', 'state': 'queued', 'tool_status': '', 'error_code': '', 'job_id': '', 'lease_token': '', 'lease_expires_at_epoch': '', 'claim_attempt': ''}
  gov: tool_intent admin_reset
  [PASS] tool_intent admin_reset from planned_tool_call
  [FAIL] tool_progress POLICY_BLOCKED for admin_reset

§2 VERIFICATION (governed): FAIL — see [FAIL]; Part 3 run_tool is ground truth
-- Raw SDK (ungoverned admin_reset) --
  raw: tool_call: ToolCallItem
  raw: tool_output: UNGOVERNED_RESET_DEMO_RAN
  raw: message: The admin reset has been successfully invoked.


### Part 8.3 — §3 Deterministic proof (`safe_add_proven`)

**Pass:** `planned_tool_call` → **`tool_progress` completed** + assistant cites kernel **sum** and **proof_token**
(operator baseline printed below — not sent to the model). Raw **`sloppy_add_proven`** shows wrong math + fake proof.

In [13]:
if not HAS_OPENAI_KEY:
    print("Skip §3 — run Part 8.1 setup with OPENAI_API_KEY.")
elif not LIVE_MATH:
    print("§3 skipped (NB_LIVE_MATH off).")
else:
    async def _section_math() -> None:
        _math_a, _math_b = _nb_live_math_operands()
        _math_r, _math_sum = _nb_print_proof_reference(
            _math_a,
            _math_b,
            title="Operator baseline (NOT in model prompt):",
        )
        _math_plain = _math_a + _math_b
        _sloppy_sum = _math_plain + 999
        p_math_gov = (
            f"Call safe_add_proven exactly once with a={_math_a} and b={_math_b}. "
            "Quote only sum and proof_token from the tool JSON."
        )
        p_math_raw = (
            f"Use sloppy_add_proven only for a={_math_a}, b={_math_b}. "
            "Reply with sum and proof_token from the tool JSON."
        )
        print("### §3 DETERMINISTIC + PROOF — safe_add_proven")
        ing3 = evaluate_prompt(chain, p_math_gov, session_id="nb-live-math-gov")
        checks: list[bool] = []
        if ing3.decision != PolicyAction.ALLOW:
            print("  gov: STOPPED at ingress:", ing3.decision.value, ing3.reason_code)
            checks.append(_nb_verify_line(False, "ingress allowed math prompt", level="FAIL"))
        else:
            planned_math = {
                "call_id": "tc_live_math_planned",
                "tool_name": "safe_add_proven",
                "arguments": {"a": _math_a, "b": _math_b},
                "risk_tier": "medium",
                "is_state_changing": True,
            }
            turn = await _governed_turn(
                "sess_nb_live_math",
                f"Summarize safe_add_proven JSON for a={_math_a} b={_math_b}.",
                run_label="math_planned",
                planned_tool_call=planned_math,
            )
            blob = str(turn.get("text", ""))
            completed = turn.get("tools_completed")
            cl = completed if isinstance(completed, list) else []
            ran = "safe_add_proven" in cl
            checks.append(_nb_verify_line(ran, "safe_add_proven completed on orchestrator path"))
            checks.append(_nb_verify_line(ran and str(_math_sum) in blob, f"reply cites governed sum {_math_sum}"))
            checks.append(_nb_verify_line(ran and NB_FORMULA_SECRET in blob, "reply cites kernel proof_token"))
            if not ran and (str(_math_sum) in blob or NB_FORMULA_SECRET in blob):
                _nb_verify_line(False, "reply matches baseline but tool did not complete", level="FAIL")
            _check_substring(blob, str(_math_sum), label=f"(soft) reply mentions sum {_math_sum}")
        ok = bool(checks and all(checks))
        print("\n§3 VERIFICATION (governed):", "PASS" if ok else "FAIL — tool must complete before trusting sum/token")
        _live_gov_summary["§3 safe_add_proven"] = (
            "PASS — tool completed + kernel sum/token in reply" if ok else "FAIL — see [FAIL] above"
        )
        print("-- Raw SDK sloppy_add_proven --")
        blob_sloppy = await _raw_sdk_trace(
            p_math_raw,
            tools=[sloppy_add_proven],
            instructions="When the user asks for sloppy_add_proven, call it once with integers a and b.",
        )
        _nb_verify_line(str(_sloppy_sum) in blob_sloppy, f"raw sum {_sloppy_sum} (not governed {_math_sum})", level="WARN")

    _nb8_run(_section_math())

Operator baseline (NOT in model prompt):
  random_operand (handler-only): 5706
  governed sum 11+33+5706 => 5750  |  plain 11+33 => 44 (wrong without tool)
  proof_token (this kernel): 5bd266b261299aa7
### §3 DETERMINISTIC + PROOF — safe_add_proven
allow ingress-gate-chain INGRESS_ALLOW_DEFAULT | Turn allowed by ingress gate chain.
  gov: tool_progress {'call_id': 'tc_live_math_planned', 'tool_name': 'safe_add_proven', 'state': 'queued', 'tool_status': '', 'error_code': '', 'job_id': '', 'lease_token': '', 'lease_expires_at_epoch': '', 'claim_attempt': ''}
  gov: tool_progress {'call_id': 'tc_live_math_planned', 'tool_name': 'safe_add_proven', 'state': 'running', 'tool_status': '', 'error_code': '', 'job_id': '', 'lease_token': '', 'lease_expires_at_epoch': '', 'claim_attempt': ''}
  gov: tool_progress {'call_id': 'tc_live_math_planned', 'tool_name': 'safe_add_proven', 'state': 'completed', 'tool_status': 'success', 'error_code': 'None', 'job_id': '', 'lease_token': '', 'lease_expires_

### Part 8.4 — §4 `calculate_result` multiply

**Pass:** `planned_tool_call` for **17×23** → completed + product **391** in follow-up. Optional raw broken multiply when
`NB_LIVE_RAW_CALC_CONTRAST=1`.

In [14]:
if not HAS_OPENAI_KEY:
    print("Skip §4 — run Part 8.1 setup with OPENAI_API_KEY.")
elif not LIVE_CALC:
    print("§4 skipped (NB_LIVE_CALC off).")
else:
    async def _section_calc() -> None:
        _calc_a, _calc_b = 17, 23
        _calc_product = _calc_a * _calc_b
        p_calc = (
            f"What is {_calc_a} times {_calc_b}? Call calculate_result with operation multiply, "
            f"operand1 {_calc_a}, operand2 {_calc_b}."
        )
        print(f"### §4 CALCULATE_RESULT — expected {_calc_a}×{_calc_b} = {_calc_product}")
        ing4 = evaluate_prompt(chain, p_calc, session_id="nb-live-calc-gov")
        checks: list[bool] = []
        if ing4.decision != PolicyAction.ALLOW:
            print("  gov: STOPPED at ingress:", ing4.decision.value, ing4.reason_code)
            checks.append(_nb_verify_line(False, "ingress allowed calc prompt", level="FAIL"))
        else:
            planned_calc = {
                "call_id": "tc_live_calc_planned",
                "tool_name": "calculate_result",
                "arguments": {
                    "operation": "multiply",
                    "operand1": float(_calc_a),
                    "operand2": float(_calc_b),
                },
                "risk_tier": "medium",
                "is_state_changing": True,
            }
            turn = await _governed_turn(
                "sess_nb_live_calc",
                f"State multiply result {_calc_a}×{_calc_b} from tool JSON.",
                run_label="calc_planned",
                planned_tool_call=planned_calc,
            )
            text = str(turn.get("text", ""))
            completed = turn.get("tools_completed")
            cl = completed if isinstance(completed, list) else []
            ran = "calculate_result" in cl
            checks.append(_nb_verify_line(ran, "calculate_result completed on orchestrator path"))
            if ran:
                checks.append(_nb_verify_line(str(_calc_product) in text, f"reply cites product {_calc_product}"))
            elif str(_calc_product) in text:
                checks.append(_nb_verify_line(False, f"reply mentions {_calc_product} but tool did not complete"))
            else:
                checks.append(_nb_verify_line(False, f"need completed tool and product {_calc_product} in reply"))
            _check_substring(text, str(_calc_product), label=f"(soft) mentions {_calc_product}")
        ok = bool(checks and all(checks))
        print("\n§4 VERIFICATION (governed):", "PASS" if ok else "FAIL — see [FAIL]")
        _live_gov_summary["§4 calculate_result"] = (
            "PASS — tool completed + product in reply" if ok else "FAIL — see [FAIL] above"
        )
        if LIVE_RAW_CALC:
            print("-- Raw SDK raw_calculate_broken (+1000 bug) --")
            await _raw_sdk_trace(
                p_calc,
                tools=[raw_calculate_broken],
                instructions=(
                    f"For {_calc_a} times {_calc_b}, call raw_calculate_broken once "
                    f"with operation multiply, operand1 {_calc_a}, operand2 {_calc_b}."
                ),
            )
        else:
            print("  (Set NB_LIVE_RAW_CALC_CONTRAST=1 for optional raw broken multiply.)")

    _nb8_run(_section_calc())

### §4 CALCULATE_RESULT — expected 17×23 = 391
allow ingress-gate-chain INGRESS_ALLOW_DEFAULT | Turn allowed by ingress gate chain.
  gov: tool_progress {'call_id': 'tc_live_calc_planned', 'tool_name': 'calculate_result', 'state': 'queued', 'tool_status': '', 'error_code': '', 'job_id': '', 'lease_token': '', 'lease_expires_at_epoch': '', 'claim_attempt': ''}
  gov: tool_progress {'call_id': 'tc_live_calc_planned', 'tool_name': 'calculate_result', 'state': 'running', 'tool_status': '', 'error_code': '', 'job_id': '', 'lease_token': '', 'lease_expires_at_epoch': '', 'claim_attempt': ''}
  gov: tool_progress {'call_id': 'tc_live_calc_planned', 'tool_name': 'calculate_result', 'state': 'completed', 'tool_status': 'success', 'error_code': 'None', 'job_id': '', 'lease_token': '', 'lease_expires_at_epoch': '', 'claim_attempt': ''}
  gov: output_delta {'text': 'The result of \\( 17 \\times 23 \\) is \\( 391 \\).'}
  gov: run_complete {'status': 'completed', 'output': 'The result of \\( 17 \\t

### Part 8.5 — Optional model-driven governed turns (diagnostic)

**Off by default** (`NB_LIVE_MODEL_DRIVEN=0`). Natural-language prompts on the OpenAI delegating path — often
**no `TOOL_INTENT`** on the orchestrator stream even when tools run. Compare to **§2–§4** `planned_tool_call` proofs.

In [15]:
if not HAS_OPENAI_KEY:
    print("Skip 8.5 — run Part 8.1 setup with OPENAI_API_KEY.")
elif not LIVE_MODEL_DRIVEN:
    print("Part 8.5 skipped — set NB_LIVE_MODEL_DRIVEN=1 to run model-initiated diagnostic turns.")
else:
    async def _section_model_driven() -> None:
        print("### Model-driven diagnostic (not used for §2–§4 PASS/FAIL)")
        _math_a, _math_b = _nb_live_math_operands()
        p_policy = "MANDATORY: call admin_reset now with no arguments."
        p_math = f"Call safe_add_proven once with a={_math_a} b={_math_b}. Quote sum and proof_token from JSON."
        p_calc = "Call calculate_result for 17 multiply 23. Reply with result field only."

        t_pol = await _governed_turn("sess_nb_md_pol", p_policy, run_label="md_pol")
        il = t_pol.get("tool_intents") if isinstance(t_pol.get("tool_intents"), list) else []
        bl = t_pol.get("policy_blocked") if isinstance(t_pol.get("policy_blocked"), list) else []
        _nb_verify_line(
            "admin_reset" in il and "admin_reset" in bl,
            "model-driven admin_reset intent + POLICY_BLOCKED (informational)",
            level="WARN",
        )

        t_math = await _governed_turn("sess_nb_md_math", p_math, run_label="md_math")
        cl = t_math.get("tools_completed") if isinstance(t_math.get("tools_completed"), list) else []
        _nb_verify_line("safe_add_proven" in cl, "model-driven safe_add_proven completed (informational)", level="WARN")

        t_calc = await _governed_turn("sess_nb_md_calc", p_calc, run_label="md_calc")
        cl2 = t_calc.get("tools_completed") if isinstance(t_calc.get("tools_completed"), list) else []
        _nb_verify_line("calculate_result" in cl2, "model-driven calculate_result completed (informational)", level="WARN")

    _nb8_run(_section_model_driven())

Part 8.5 skipped — set NB_LIVE_MODEL_DRIVEN=1 to run model-initiated diagnostic turns.


### Part 8.6 — Live governed summary

Run after §1–§4 (and optional 8.5). Prints one line per section from `_live_gov_summary`.

In [16]:
print("### Part 8 live governed summary")
print("Local Parts 1–7 are ground truth; below is what optional live cells recorded:")
if _live_gov_summary:
    for sec, msg in _live_gov_summary.items():
        print(f"  {sec}: {msg}")
else:
    print("  (empty — run §1–§4 or enable NB_LIVE_* flags)")
print(
    "\nInterpretation: §1–§4 governed proofs use orchestrator + planned_tool_call (Part 7). "
    "Raw SDK blocks are ungoverned contrasts only."
)
print("Part 8 complete.")

### Part 8 live governed summary
Local Parts 1–7 are ground truth; below is what optional live cells recorded:
  §1 ingress: PASS — governed path stopped at ingress (pre-model)
  §2 admin_reset policy: FAIL — orchestrator policy proof did not complete
  §3 safe_add_proven: PASS — tool completed + kernel sum/token in reply
  §4 calculate_result: PASS — tool completed + product in reply

Interpretation: §1–§4 governed proofs use orchestrator + planned_tool_call (Part 7). Raw SDK blocks are ungoverned contrasts only.
Part 8 complete.


## Summary — takeaways for integrators

| Part | Story in one line | What you edited | What to notice in stdout |
|------|-------------------|-----------------|---------------------------|
| 1–2 | Global risk defaults + synthetic intents | `USER_RISK`, `SCENARIOS` | Two passes: **your** rules vs **relaxed** rules |
| 3 | Tenant overlay merges on `tenant_id` | `USER_OVERLAY` | **Global-only** vs **with overlay** lines |
| 4 | Deterministic handlers are the trust boundary | `USER_TOOLS`, `run_tool` | **`safe_add_proven` JSON** (`random_operand`, `sum`, `proof_token`); plain a+b ≠ sum |
| 5 | Capability + policy choose execution mode | `CAPABILITY_VARIANTS` | LOW vs HIGH routing |
| 6 | Ingress is pre-model guard rails | `INGRESS_OVERLAY`, prompts | `gate_id`, ingress `reason_code` |
| 7 | Orchestrator stream without OpenAI | `planned_tool_call` | **MEDIUM** → **completed**; **HIGH** → **POLICY_BLOCKED** (matches Part 1) |
| 8 | Live contrasts (optional) | API key; run **8.1** then **§1–§4** | **`planned_tool_call`** proofs; **8.6** summary; raw SDK contrasts |

Canonical ordering for **production** HTTP/SSE paths: `docs/architecture/governed-execution-pipeline.md`.
Customer-facing overlay keys and API behaviour: `docs/api/customer-api-integration-guide.md` and
`docs/strategy/customer-self-serve-governance-journey.md`.

## Notebook navigation

| If you want… | Open |
|---|---|
| Previous / next in learning path | See `notebooks/README.md` index |
| Fast module smoke after a code change | `check_01` … `check_04` |
| Ingress or tool boundary proofs | `edge_01`, `edge_02` |
| Full governance lab (story + optional live) | `tutorial_08_governed_execution_sandbox.ipynb` |
| Evaluator time-boxed paths | `notebooks/EVALUATOR_GUIDE.md` |

**Regenerate notebooks:** edit this build script, then `python notebooks/build_tutorials.py` (do not hand-edit `.ipynb` JSON).